# SI Notebook 3: VbDriver active-space diagnostics

This supplementary notebook accompanies the VB section of the article. It demonstrates the direct `VbDriver` API for H2, ethylene pi-bond resonance, and the benzene equivalent-center VBSCF stabilization checkpoint. The notebook is intentionally small; the source tests remain the authoritative regression layer.

In [1]:
from pathlib import Path
from contextlib import contextmanager, redirect_stdout, redirect_stderr
import importlib.util
import io
import sys

from veloxchem.molecule import Molecule
from veloxchem.molecularbasis import MolecularBasis
from veloxchem.scfrestdriver import ScfRestrictedDriver

start = Path.cwd().resolve()
repo_root = next(path for path in (start, *start.parents) if (path / 'src' / 'pymodule').exists())
for module_name in ('orbitalanalyzerdriver', 'vbdriver'):
    module_path = repo_root / 'src' / 'pymodule' / f'{module_name}.py'
    spec = importlib.util.spec_from_file_location(f'veloxchem.{module_name}', module_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f'veloxchem.{module_name}'] = module
    spec.loader.exec_module(module)

from veloxchem.vbdriver import VbComputeOptions, VbDriver

@contextmanager
def quiet_veloxchem():
    with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
        yield

def run_rhf(molecule, basis_label='sto-3g'):
    basis = MolecularBasis.read(molecule, basis_label, ostream=None)
    scf = ScfRestrictedDriver()
    scf.ostream.mute()
    scf.xcfun = 'hf'
    with quiet_veloxchem():
        scf.compute(molecule, basis)
    return basis, scf

def summarize_vb(label, results):
    active_space = results.get('active_space')
    diagnostics = results.get('diagnostics', {})
    return {
        'case': label,
        'energy_hartree': float(results.get('energy', 0.0)),
        'weights': [round(float(weight), 6) for weight in results.get('weights', [])],
        'active_candidate': None if active_space is None else active_space.active_candidate_label,
        'active_structures': None if active_space is None else [structure.label for structure in active_space.structures],
        'vb_method': diagnostics.get('vb_method'),
        'message': diagnostics.get('message'),
    }

vb_source = repo_root / 'src' / 'pymodule' / 'vbdriver.py'
print(f'Using local VB source from {vb_source}')

Using local VB source from /home/linares/app/VeloxChem/src/pymodule/vbdriver.py


## H2 three-structure model

H2 is the smallest transparent VB model. The same active bond is evaluated in VBCI, VBSCF, and BOVB mode so the notebook can show the method ladder used throughout the article.

In [2]:
h2 = Molecule.read_str('''
H 0.00000000 0.00000000 0.00000000
H 0.74000000 0.00000000 0.00000000
''')
h2.set_charge(0)
h2.set_multiplicity(1)
h2_basis, h2_scf = run_rhf(h2, basis_label='sto-3g')

h2_rows = []
for mode in ('vbci', 'vbscf', 'bovb'):
    options = VbComputeOptions(
        mode=mode,
        active_bond=(0, 1),
        active_candidate_subtype='sigma',
        optimize_orbitals=(mode == 'vbscf'),
        include_bovb=(mode == 'bovb'),
        include_ionic=True,
        freeze_inactive_orbitals=True,
    )
    result = VbDriver().compute(h2, h2_basis, reference_orbitals=h2_scf.mol_orbs, options=options)
    h2_rows.append(summarize_vb(f'H2 {mode.upper()}', result))

h2_rows

                                                                                                                          
                                            Self Consistent Field Driver Setup                                            
                                                                                                                          
                   Wave Function Model             : Spin-Restricted Hartree-Fock                                         
                   Initial Guess Model             : Superposition of Atomic Densities                                    
                   Convergence Accelerator         : Two Level Direct Inversion of Iterative Subspace                     
                   Max. Number of Iterations       : 50                                                                   
                   Max. Number of Error Vectors    : 10                                                                   
                

[{'case': 'H2 VBCI',
  'energy_hartree': -1.137283834651965,
  'weights': [0.611829, 0.194086, 0.194086],
  'active_candidate': 'BD_sigma_1',
  'active_structures': ['covalent',
   'ionic_A_minus_B_plus',
   'ionic_A_plus_B_minus'],
  'vb_method': 'vbci',
  'message': 'Two-electron VB-CI result from spin-adapted singlet structures.'},
 {'case': 'H2 VBSCF',
  'energy_hartree': -1.1372838346519645,
  'weights': [0.611829, 0.194086, 0.194086],
  'active_candidate': 'BD_sigma_1',
  'active_structures': ['covalent',
   'ionic_A_minus_B_plus',
   'ionic_A_plus_B_minus'],
  'vb_method': 'vbscf',
  'message': 'H2 VB-SCF result with common symmetry-preserving breathing orbitals'},
 {'case': 'H2 BOVB',
  'energy_hartree': -1.1372838346519658,
  'weights': [0.611829, 0.194086, 0.194086],
  'active_candidate': 'BD_sigma_1',
  'active_structures': ['covalent',
   'ionic_A_minus_B_plus',
   'ionic_A_plus_B_minus'],
  'vb_method': 'bovb',
  'message': 'H2 BOVB result with structure-specific breathing

## Ethylene pi-bond active space

Ethylene demonstrates how an organic pi candidate selected from the shared analyzer payload becomes a two-center VB active space. The result exposes the selected candidate, structure labels, energy, weights, and diagnostics.

In [3]:
ethylene = Molecule.read_str('''
C -0.66950000  0.00000000 0.00000000
C  0.66950000  0.00000000 0.00000000
H -1.23210000  0.92890000 0.00000000
H -1.23210000 -0.92890000 0.00000000
H  1.23210000  0.92890000 0.00000000
H  1.23210000 -0.92890000 0.00000000
''')
ethylene.set_charge(0)
ethylene.set_multiplicity(1)
ethylene_basis, ethylene_scf = run_rhf(ethylene, basis_label='sto-3g')

ethylene_vbci = VbDriver().compute(
    ethylene,
    ethylene_basis,
    reference_orbitals=ethylene_scf.mol_orbs,
    options=VbComputeOptions(
        mode='vbci',
        active_bond=(0, 1),
        active_candidate_subtype='pi',
        optimize_orbitals=False,
        include_ionic=True,
        freeze_inactive_orbitals=True,
    ),
)

ethylene_vbscf = VbDriver().compute(
    ethylene,
    ethylene_basis,
    reference_orbitals=ethylene_scf.mol_orbs,
    options=VbComputeOptions(
        mode='vbscf',
        active_bond=(0, 1),
        active_candidate_subtype='pi',
        optimize_orbitals=True,
        include_ionic=True,
        freeze_inactive_orbitals=True,
        orbital_relaxation_symmetry='equivalent-centers',
        orbital_amplitude_bound=0.05,
    ),
)

[summarize_vb('ethylene VBCI', ethylene_vbci), summarize_vb('ethylene VBSCF', ethylene_vbscf)]

                                                                                                                          
                                            Self Consistent Field Driver Setup                                            
                                                                                                                          
                   Wave Function Model             : Spin-Restricted Hartree-Fock                                         
                   Initial Guess Model             : Superposition of Atomic Densities                                    
                   Convergence Accelerator         : Two Level Direct Inversion of Iterative Subspace                     
                   Max. Number of Iterations       : 50                                                                   
                   Max. Number of Error Vectors    : 10                                                                   
                

[{'case': 'ethylene VBCI',
  'energy_hartree': -77.07958112029011,
  'weights': [0.554472, 0.222764, 0.222764],
  'active_candidate': 'BD_pi_4',
  'active_structures': ['covalent',
   'ionic_A_minus_B_plus',
   'ionic_A_plus_B_minus'],
  'vb_method': 'vbci',
  'message': 'Two-electron VB-CI result from spin-adapted singlet structures.'},
 {'case': 'ethylene VBSCF',
  'energy_hartree': -77.07958112029011,
  'weights': [0.554472, 0.222764, 0.222764],
  'active_candidate': 'BD_pi_4',
  'active_structures': ['covalent',
   'ionic_A_minus_B_plus',
   'ionic_A_plus_B_minus'],
  'vb_method': 'vbscf',
  'message': 'H2 VB-SCF result with common symmetry-preserving breathing orbitals'}]

## Benzene equivalent-center stabilization

The article identifies benzene as a stabilization checkpoint for the current VBSCF implementation. The equivalent-center option constrains the pi-system relaxation to a conservative symmetry-preserving path before any BOVB extension is attempted.

In [4]:
benzene = Molecule.read_str('''
C  1.39700000  0.00000000 0.00000000
C  0.69850000  1.20983700 0.00000000
C -0.69850000  1.20983700 0.00000000
C -1.39700000  0.00000000 0.00000000
C -0.69850000 -1.20983700 0.00000000
C  0.69850000 -1.20983700 0.00000000
H  2.48100000  0.00000000 0.00000000
H  1.24050000  2.14883000 0.00000000
H -1.24050000  2.14883000 0.00000000
H -2.48100000  0.00000000 0.00000000
H -1.24050000 -2.14883000 0.00000000
H  1.24050000 -2.14883000 0.00000000
''')
benzene.set_charge(0)
benzene.set_multiplicity(1)
benzene_basis, benzene_scf = run_rhf(benzene, basis_label='sto-3g')

benzene_vbscf = VbDriver().compute(
    benzene,
    benzene_basis,
    reference_orbitals=benzene_scf.mol_orbs,
    options=VbComputeOptions(
        mode='vbscf',
        active_pi_atoms=tuple(range(6)),
        active_electron_count=6,
        active_spin='singlet',
        optimize_orbitals=True,
        include_ionic=True,
        freeze_inactive_orbitals=True,
        orbital_relaxation_symmetry='equivalent-centers',
        orbital_amplitude_bound=0.05,
    ),
)

summary = summarize_vb('benzene equivalent-center VBSCF', benzene_vbscf)
summary['diagnostic_keys'] = sorted(benzene_vbscf.get('diagnostics', {}).keys())
summary

                                                                                                                          
                                            Self Consistent Field Driver Setup                                            
                                                                                                                          
                   Wave Function Model             : Spin-Restricted Hartree-Fock                                         
                   Initial Guess Model             : Superposition of Atomic Densities                                    
                   Convergence Accelerator         : Two Level Direct Inversion of Iterative Subspace                     
                   Max. Number of Iterations       : 50                                                                   
                   Max. Number of Error Vectors    : 10                                                                   
                

{'case': 'benzene equivalent-center VBSCF',
 'energy_hartree': -224.9830800128837,
 'weights': [0.000875,
  0.006382,
  0.009003,
  0.003931,
  0.000787,
  6.3e-05,
  0.000659,
  6.2e-05,
  0.000541,
  0.003099,
  0.002904,
  0.002788,
  0.001494,
  0.00735,
  0.000306,
  0.006963,
  0.000372,
  0.000637,
  0.000272,
  0.000631,
  0.006382,
  0.002129,
  0.017608,
  0.00854,
  4e-06,
  0.003617,
  0.000519,
  0.001906,
  0.000768,
  0.000766,
  0.000114,
  0.003319,
  0.001951,
  0.000583,
  0.00349,
  0.000216,
  0.001553,
  0.003489,
  0.000531,
  0.00082,
  0.009003,
  0.017608,
  0.018527,
  0.032437,
  0.004628,
  0.006603,
  0.008904,
  0.000284,
  7.6e-05,
  0.001134,
  0.020546,
  0.007132,
  0.034464,
  0.005928,
  6.6e-05,
  0.014712,
  0.004014,
  0.000127,
  0.008084,
  9.8e-05,
  0.003931,
  0.00854,
  0.032437,
  0.006319,
  0.0,
  0.001497,
  1.1e-05,
  0.00036,
  1.4e-05,
  0.00057,
  0.0038,
  0.010446,
  0.007111,
  0.000523,
  0.005253,
  0.012467,
  8.5e-05,
  0.000